<a href="https://colab.research.google.com/github/Bassendiaye/mes_notebooks/blob/main/2_Representation_donnees_RI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Représentation des données et recherche d'information

Nous allons discuter les points suivants:


* Stockage des données
* Modèle du sac de mots (bag of words)
* Matrice Documents x Termes
* Pré-traitements
* Extraction des traits (ou features)
* Schémas de pondération (TF, TF-IDF...)

# Stockage de données

Tout d'abord, nous stockons notre petit corpus sous forme de liste de documents.

In [ ]:
corpus = ["Le cinéma est un art, c’est aussi une industrie.",
        "Personne, quand il est petit, ne veut être critique de cinéma. Mais ensuite, en France, tout le monde a un deuxième métier : critique de cinéma !",
        "Tout le monde a des rêves de Hollywood.",
        "C'est la crise, l'économie de la France est menacée par la mondialisation.",
        "En temps de crise, reconstruire l'industrie : tout un art!",
         "Quand une usine ferme, c'est que l'économie va mal."]

Nous stockons nos documents et les classes correspondantes sous forme de dictionnaire Python

In [ ]:
data = {'Text':corpus}

Nous importons la bibliothèque **pandas** qui permet la manipulation et l'analyse des données. Elle propose en particulier des structures de données et des opérations de manipulation de tableaux numériques.

La structure **DataFrame** permet le stockage des données en 2 dimensions: lignes et colonnes

In [ ]:
import pandas as pd
from pandas import DataFrame

In [ ]:
df = pd.DataFrame(data)

In [ ]:
df

,Text
0,"Le cinéma est un art, c’est aussi une industrie."
1,"Personne, quand il est petit, ne veut être cri..."
2,Tout le monde a des rêves de Hollywood.
3,"C'est la crise, l'économie de la France est me..."
4,"En temps de crise, reconstruire l'industrie : ..."
5,"Quand une usine ferme, c'est que l'économie va..."


# Pré-traitement
Pour le pré-traitement nous utilisons spacy.

Nous importons spacy

In [ ]:
import spacy

Nous utilisons le modèle français (petit modele "small") entrainé sur des articles d'actualités.

In [ ]:
nlp = spacy.load("fr_core_news_sm")

## Tokenisation
Nous parcourons les textes et appliquons la segmentation des mots sur chaque texte.

In [ ]:
for text in df['Text']:
    doc = nlp(text)
    for token in doc:
        print(token.text)

Le
cinéma
est
un
art
,
c’
est
aussi
une
industrie
.
Personne
,
quand
il
est
petit
,
ne
veut
être
critique
de
cinéma
.
Mais
ensuite
,
en
France
,
tout
le
monde
a
un
deuxième
métier
:
critique
de
cinéma
!
Tout
le
monde
a
des
rêves
de
Hollywood
.
C'
est
la
crise
,
l'
économie
de
la
France
est
menacée
par
la
mondialisation
.
En
temps
de
crise
,
reconstruire
l'
industrie
:
tout
un
art
!
Quand
une
usine
ferme
,
c'
est
que
l'
économie
va
mal
.


## Suppression des mots vides

Nous importons la liste des mots vides

In [ ]:
french_stopwords = spacy.lang.fr.stop_words.STOP_WORDS

In [ ]:
# Ajouter quelques mots vides
french_stopwords.add('mal')
french_stopwords.add('ensuite')
french_stopwords.add('veut')
french_stopwords.add('jamais')

Nous créons une liste de phrases pre-traitées (tokenisées et sans mots vides).

In [ ]:
#Liste de phrases filtrées
phrases_pretraitees=[]

def is_stop_word(word, stop_list):
    return str(word).lower() in stop_list

for text in df['Text']:
    doc = nlp(text)
    # filtrer les stop words
    token_list = []
    for word in doc:
        if not is_stop_word(word,french_stopwords) and word.is_punct==False:
            token_list.append(word.text)
    phrases_pretraitees.append(' '.join(token_list))

print("Liste des phrases pré-traitées:", phrases_pretraitees)

Liste des phrases pré-traitées: ['cinéma art industrie', 'petit critique cinéma France monde métier critique cinéma', 'monde rêves Hollywood', 'crise économie France menacée mondialisation', 'temps crise reconstruire industrie art', 'usine ferme économie']


# Extraction de caractéristiques/traits (*features*)

Pour l'extraction des traits, nous allons utiliser la bibliothèque **sklearn** qui fournit des classes pour la transformation des données.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer,TfidfTransformer,TfidfVectorizer

La bibliothèque de sklearn contient une fonction **CountVectorizer()** qui permet de "vectoriser" un ensemble de textes bruts en prenant en compte un certain nombre de prétraitements (preprocessing) couramment employés.
Le CountVectorizer permet de compter le nombre de mots (fréquence des termes).

Nous créons d'abord une représentation **booléenne** en ajoutant le paramètre **binary=True** au CountVectorizer()

In [ ]:
cv_bool = CountVectorizer(binary=True)

La fonction **fit(textes)** de CountVectorizer construit un dictionnaire de vocabulaire contenu dans les textes

In [ ]:
cv_bool.fit(phrases_pretraitees)

CountVectorizer(binary=True)

Observons le vocabulaire automatiquement construit à partir de ces textes :

In [ ]:
cv_bool.vocabulary_

{'cinéma': 1,
 'art': 0,
 'industrie': 7,
 'petit': 12,
 'critique': 3,
 'france': 5,
 'monde': 9,
 'métier': 11,
 'rêves': 14,
 'hollywood': 6,
 'crise': 2,
 'économie': 17,
 'menacée': 8,
 'mondialisation': 10,
 'temps': 15,
 'reconstruire': 13,
 'usine': 16,
 'ferme': 4}

On peut maintenant construire la matrice documents * termes (dtm) à l'aide de la fonction **transform()** de CountVectorizer.

In [ ]:
dtm_bool = cv_bool.transform(phrases_pretraitees)

Vérifier les dimensions de notre matrice.

In [ ]:
dtm_bool.shape

(6, 18)

Dimension de la matrice: 6x21 (6 documents; 18 mots)

Nous jetons un coup d'oeil au contenu de la matrice :

In [ ]:
dtm_bool.toarray()

array([[1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0],
       [0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1],
       [1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1]])

Pour accéder au nom des termes du vocabulaire,on peut utiliser la fonction **get_feature_names_out()**.

In [ ]:
terms = cv_bool.get_feature_names_out()

In [ ]:
terms

array(['art', 'cinéma', 'crise', 'critique', 'ferme', 'france',
       'hollywood', 'industrie', 'menacée', 'monde', 'mondialisation',
       'métier', 'petit', 'reconstruire', 'rêves', 'temps', 'usine',
       'économie'], dtype=object)

Nous pouvons créer un DataFrame pour représenter les textes et les attributs.

In [ ]:
df_cv_bool = DataFrame(dtm_bool.toarray(), columns=terms)
df_cv_bool

,art,cinéma,crise,critique,ferme,france,hollywood,industrie,menacée,monde,mondialisation,métier,petit,reconstruire,rêves,temps,usine,économie
0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,1,0,1,0,1,0,0,0,1,0,1,1,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0
3,0,0,1,0,0,1,0,0,1,0,1,0,0,0,0,0,0,1
4,1,0,1,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0
5,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1


Au lieu d'appeler les fonctions fit() et transform() successivement, c'est possible de faire juste un appel de fonction avec **fit_transform()**. Cette dernière apprend le dictionnaire de vocabulaire contenu dans les textes et retourne la matrice sous forme de document-terme.

La matrice suivante dtm_bool2 et la matrice précédente dtm_bool sont identique.

In [ ]:
# Dataframe
dtm_bool2 = cv_bool.fit_transform(phrases_pretraitees)
dtm_bool2

<6x18 sparse matrix of type '<class 'numpy.int64'>'
	with 25 stored elements in Compressed Sparse Row format>

Maintenant, nous allons utiliser le même procédé avec le CountVectorizer pour créer une matrice basée sur les **fréquences des termes** (TF).

In [ ]:
cv_tf = CountVectorizer()
dtm_tf = cv_tf.fit_transform(phrases_pretraitees)
terms = cv_tf.get_feature_names_out()
df_tf = DataFrame(dtm_tf.toarray(), columns=terms)
df_tf

,art,cinéma,crise,critique,ferme,france,hollywood,industrie,menacée,monde,mondialisation,métier,petit,reconstruire,rêves,temps,usine,économie
0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,2,0,2,0,1,0,0,0,1,0,1,1,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0
3,0,0,1,0,0,1,0,0,1,0,1,0,0,0,0,0,0,1
4,1,0,1,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0
5,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1


## Représentation TFxIDF

A la place du nombre d'occurrences (TF), on peut utiliser la mesure TF-IDF qui prend en compte la rareté d'un mot dans le corpus :

$tf_{t,d}\times idf_t$ avec
* $tf_{t,d}$ = le nombre d'occurrences du terme $t$ dans le document $d$
* $idf_t=log\frac{N}{df_t}$ (N=nombre total de documents dans notre corpus; df_t=le nombre de documents dans lesquels $t$ apparaît)

Pour obtenir les scores tf-idf de notre corpus, nous allons utiliser **Tfidfvectorizer()**. Elle permet de calculer le nombre de mots, les valeurs idf et tf-idf en une seule étape.

In [ ]:
tfidf_vect = TfidfVectorizer()

In [ ]:
dtm_tfidf = tfidf_vect.fit_transform(phrases_pretraitees)

In [ ]:
print(dtm_tfidf.toarray())

[[0.57735027 0.57735027 0.         0.         0.         0.
  0.         0.57735027 0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.        ]
 [0.         0.51772922 0.         0.63136608 0.         0.25886461
  0.         0.         0.         0.25886461 0.         0.31568304
  0.31568304 0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.61171251 0.         0.         0.50161301 0.         0.
  0.         0.         0.61171251 0.         0.         0.        ]
 [0.         0.         0.40912489 0.         0.         0.40912489
  0.         0.         0.49892408 0.         0.49892408 0.
  0.         0.         0.         0.         0.         0.40912489]
 [0.40912489 0.         0.40912489 0.         0.         0.
  0.         0.40912489 0.         0.         0.         0.
  0.         0.49892408 0.         0.49892408 0.         0.        ]
 [0.         0.         0.     

In [ ]:
terms = tfidf_vect.get_feature_names_out()
df_tfidf = DataFrame(dtm_tfidf.toarray(), columns=terms)
df_tfidf

,art,cinéma,crise,critique,ferme,france,hollywood,industrie,menacée,monde,mondialisation,métier,petit,reconstruire,rêves,temps,usine,économie
0,0.577350,0.577350,0.000000,0.000000,0.000000,0.000000,0.000000,0.577350,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.517729,0.000000,0.631366,0.000000,0.258865,0.000000,0.000000,0.000000,0.258865,0.000000,0.315683,0.315683,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.611713,0.000000,0.000000,0.501613,0.000000,0.000000,0.000000,0.000000,0.611713,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.409125,0.000000,0.000000,0.409125,0.000000,0.000000,0.498924,0.000000,0.498924,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.409125
4,0.409125,0.000000,0.409125,0.000000,0.000000,0.000000,0.000000,0.409125,0.000000,0.000000,0.000000,0.000000,0.000000,0.498924,0.000000,0.498924,0.000000,0.000000
5,0.000000,0.000000,0.000000,0.000000,0.611713,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.611713,0.501613


Nous pouvons écrire une fonction nous permettant de créer directement une matrice à partir d'une liste de documents et d'un certain type de vectorizer.

In [ ]:
def create_matrix(sent_list,vectorizer):
    dtm = vectorizer.fit_transform(sent_list)
    df = DataFrame(dtm.toarray(),columns=vectorizer.get_feature_names_out())
    return df

In [ ]:
create_matrix(phrases_pretraitees,tfidf_vect)

,art,cinéma,crise,critique,ferme,france,hollywood,industrie,menacée,monde,mondialisation,métier,petit,reconstruire,rêves,temps,usine,économie
0,0.577350,0.577350,0.000000,0.000000,0.000000,0.000000,0.000000,0.577350,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.517729,0.000000,0.631366,0.000000,0.258865,0.000000,0.000000,0.000000,0.258865,0.000000,0.315683,0.315683,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.611713,0.000000,0.000000,0.501613,0.000000,0.000000,0.000000,0.000000,0.611713,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.409125,0.000000,0.000000,0.409125,0.000000,0.000000,0.498924,0.000000,0.498924,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.409125
4,0.409125,0.000000,0.409125,0.000000,0.000000,0.000000,0.000000,0.409125,0.000000,0.000000,0.000000,0.000000,0.000000,0.498924,0.000000,0.498924,0.000000,0.000000
5,0.000000,0.000000,0.000000,0.000000,0.611713,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.611713,0.501613


# Calcul de Distance

La bibliothèque **scipy** contient un certain nombre de fonctions utiles de calcul scientifique en Python. Nous pouvons l'utiliser pour calculer les distances entre vecteurs.

In [ ]:
from scipy.spatial import distance

Nous allons egalement utiliser **numpy** qui permet d'effectuer des calculs numériques avec Python. Elle introduit une gestion facilitée des tableaux de nombres.

In [ ]:
import numpy as np
import math

Pour tester les mesures de distance, nous allons construire le dataframe df_cv_bool qui fournit une représentation booléenne de nos données

In [ ]:
df_cv_bool

,art,cinéma,crise,critique,ferme,france,hollywood,industrie,menacée,monde,mondialisation,métier,petit,reconstruire,rêves,temps,usine,économie
0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,1,0,1,0,1,0,0,0,1,0,1,1,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0
3,0,0,1,0,0,1,0,0,1,0,1,0,0,0,0,0,0,1
4,1,0,1,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0
5,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1


Nous allons extraire les deux premiers vecteurs *a* et *b* de cette matrice.

In [ ]:
# two vectors
a = df_cv_bool.iloc[0,:-1].values
print("vector a: ", a)
b = df_cv_bool.iloc[1,:-1].values
print("vector b: ", b)

vector a:  [1 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0]
vector b:  [0 1 0 1 0 1 0 0 0 1 0 1 1 0 0 0 0]


Pour calculer la distance de Manhattan entre deux points nous pouvons utiliser la fonction **distance.cityblock()** de la bibliothèque scipy.spatial.

In [ ]:
# Mahattan distance b/w a and b
man_dist = distance.cityblock(a, b)
print("Distance Manhattan: ", man_dist)

Distance Manhattan:  7


De même, la fonction *euclidean* renvoie la distance euclidienne entre *a* et *b*.

In [ ]:
# Distance euclidienne
eucl_dist = distance.euclidean(a, b)
print("Distance Euclidienne: ", eucl_dist)

Distance Euclidienne:  2.6457513110645907


La fonction **dot(a,b)** de numpy permet d'obtenir le produit scalaire de deux vecteurs *a* et *b*.

In [ ]:
# Produit scalaire
scalar_product = np.dot(a, b, out = None)
print("Produit scalaire des deux vecteurs a et b: ", scalar_product)


Produit scalaire des deux vecteurs a et b:  1


Avec la fonction **linalg.norm(x)**, nous obtenons la norme d'un vecteur *x*.

In [ ]:
# Norme des vecteurs
vec_norm_a = np.linalg.norm(a)
vec_norm_b = np.linalg.norm(b)
print("Vector norm a: ", np.round(vec_norm_a,2))
print("Vector norm b: ", np.round(vec_norm_b,2))

Vector norm a:  1.73
Vector norm b:  2.45


# Calcul de similarité

En combinant le produit scalaire des deux vecteur avec les normes de chaque vecteur nous pouvons aisément calculer les similarités cosinus, Dice, Jaccard.

## Calculer la similarité cosinus

In [ ]:
sim_cos = scalar_product / (vec_norm_a * vec_norm_b)
print("Simalrité cosinus des vecteurs a et b: ", np.round(sim_cos,2))

Simalrité cosinus des vecteurs a et b:  0.24


## Coefficient de Dice

In [ ]:
dice = scalar_product / (vec_norm_a + vec_norm_b)
print("Dice coefficient des vecteurs a et b: ", np.round(dice,2))

Dice coefficient des vecteurs a et b:  0.24


# Calcul de similarité entre notre corpus et une requête

Pour les calcul de distance et de similarité, nous ajoutons la requête au corpus

Requête: **Pendant la crise, l'usine à rêves Hollywood critique le cynisme de l'industrie.**

In [ ]:
# notre requête
requete = "Pendant la crise, l'usine à rêves Hollywood critique le cynisme de l'industrie."

Nous créons ensuite un ensemble de tous les termes qui apparaissent dans notre corpus.
Nous en avons besoin pour pré-traiter notre requête en éliminant tous les termes présents dans la requête et qui n'apparaissent pas dans le corpus.
Nous obtenons cet ensemble grâce à la classe CountVectorizer().

In [ ]:
cv = CountVectorizer()
dtm = cv.fit_transform(phrases_pretraitees)
#Vocabulaire du corpus
corpus_vocab = cv.get_feature_names_out()
corpus_vocab

array(['art', 'cinéma', 'crise', 'critique', 'ferme', 'france',
       'hollywood', 'industrie', 'menacée', 'monde', 'mondialisation',
       'métier', 'petit', 'reconstruire', 'rêves', 'temps', 'usine',
       'économie'], dtype=object)

Ensuite, nous traitons la requête de la même manière que les documents du corpus: on élimine les mots vides et les signes de ponctuation. Mais, au-delà nous supprimons les mots de la requête qui n'apparaissent pas dans le corpus.

In [ ]:
# Liste des termes à retenir de la requête
requete_pretraitee=[]

# Traiter la requête
doc = nlp(requete)

# filtering stop words
token_list = []
for word in doc:
    # voir si le terme apparait aussi dans le corpus
    if not is_stop_word(word,french_stopwords) and word.is_punct==False and str(word.text).lower() in corpus_vocab:
        token_list.append(word.text)
requete_pretraitee.append(' '.join(token_list))

print("Requête pré-traitée:",requete_pretraitee)

Requête pré-traitée: ['crise usine rêves Hollywood critique industrie']


In [ ]:
# On ajoute les termes de la requête pré-traitée à la liste des termes du corpus
phrases_pretraitees.append(' '.join(requete_pretraitee))
print("Phrases et requête pré-traitées:",phrases_pretraitees)

Phrases et requête pré-traitées: ['cinéma art industrie', 'petit critique cinéma France monde métier critique cinéma', 'monde rêves Hollywood', 'crise économie France menacée mondialisation', 'temps crise reconstruire industrie art', 'usine ferme économie', 'crise usine rêves Hollywood critique industrie']


Maintenant nous créons une matrice de terme et de documents qui va intégrer la requête.

In [ ]:
cv = CountVectorizer(binary=True)
boolean=1
dtm = cv.fit_transform(phrases_pretraitees)
df = DataFrame(dtm.toarray(), columns=cv.get_feature_names_out())
df

,art,cinéma,crise,critique,ferme,france,hollywood,industrie,menacée,monde,mondialisation,métier,petit,reconstruire,rêves,temps,usine,économie
0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,1,0,1,0,1,0,0,0,1,0,1,1,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0
3,0,0,1,0,0,1,0,0,1,0,1,0,0,0,0,0,0,1
4,1,0,1,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0
5,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1
6,0,0,1,1,0,0,1,1,0,0,0,0,0,0,1,0,1,0


# Mesures de similarité

## Dice coefficient

In [ ]:
# calcule le nombre d’attributs communs à arr1 et arr2
def common_ones(arr1, arr2, n):
    common_att = 0
    for i in range(n):
        if (arr1[i] == 1 and arr2[i] == 1):
            common_att += 1
    return common_att

# calcule le nombre de '1' dans arr
def one_occurrences(arr, n):
    count = 0
    for i in range(n):
        if (arr[i] == 1):
            count += 1
    return count



In [ ]:
# vecteur de la requête
request = df.iloc[6,:].values
n = len(request)
n_occur_1 = one_occurrences(request, n)

if(boolean):
    for i in range(6):
        # vecteur de chaque texte
        doc_vector = df.iloc[i,:].values
        print("req: ", request)
        print("Doc: ", doc_vector)
        common_att = common_ones(request,doc_vector, n)
        n_occur_2 = one_occurrences(doc_vector, n)
        dice = common_att / (n_occur_1 + n_occur_2)
        print(f"Similarité dice coefficient entre Text #{i+1} et la requête est de: {dice:.3f}.")
else:
    for i in range(5):
        # vecteur de chaque texte
        doc_vector = df.iloc[i,:].values
        print("req: ", request)
        print("Doc: ", doc_vector)
        scalar_product = np.dot(doc_vector, request, out = None)
        vec_norm_a = np.linalg.norm(doc_vector)
        vec_norm_b = np.linalg.norm(request)
        dice = scalar_product / (vec_norm_a + vec_norm_b)
        print(f"Similarité dice coefficient entre Text #{i+1} et la requête est de: {dice:.3f}.")

req:  [0 0 1 1 0 0 1 1 0 0 0 0 0 0 1 0 1 0]
Doc:  [1 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0]
Similarité dice coefficient entre Text #1 et la requête est de: 0.111.
req:  [0 0 1 1 0 0 1 1 0 0 0 0 0 0 1 0 1 0]
Doc:  [0 1 0 1 0 1 0 0 0 1 0 1 1 0 0 0 0 0]
Similarité dice coefficient entre Text #2 et la requête est de: 0.083.
req:  [0 0 1 1 0 0 1 1 0 0 0 0 0 0 1 0 1 0]
Doc:  [0 0 0 0 0 0 1 0 0 1 0 0 0 0 1 0 0 0]
Similarité dice coefficient entre Text #3 et la requête est de: 0.222.
req:  [0 0 1 1 0 0 1 1 0 0 0 0 0 0 1 0 1 0]
Doc:  [0 0 1 0 0 1 0 0 1 0 1 0 0 0 0 0 0 1]
Similarité dice coefficient entre Text #4 et la requête est de: 0.091.
req:  [0 0 1 1 0 0 1 1 0 0 0 0 0 0 1 0 1 0]
Doc:  [1 0 1 0 0 0 0 1 0 0 0 0 0 1 0 1 0 0]
Similarité dice coefficient entre Text #5 et la requête est de: 0.182.
req:  [0 0 1 1 0 0 1 1 0 0 0 0 0 0 1 0 1 0]
Doc:  [0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 1]
Similarité dice coefficient entre Text #6 et la requête est de: 0.111.


Observation: Le texte #3 est le plus proche de la requête.

Observation: Le texte #3 est le plus proche de la requête.

## Similarité cosinus

In [ ]:
# vecteur de la requête
request = df.iloc[6,:].values

for i in range(6):
    # vecteur de chaque texte
    doc_vector = df.iloc[i,:].values
    # distance cosinus entre les deux vecteurs
    cos_dist = distance.cosine(doc_vector.tolist(), request.tolist())
    sim_cos = 1 - cos_dist
    # afficher le resultat
    print(f"Similarité cosinus entre Text #{i+1} et la requête est de: {sim_cos:.3f}.")

Observation: Le texte #3 est le plus proche de la requête.